# KAN-Vim — Chạy thí nghiệm (Colab T4)
1. Upload `kan-vim.zip` mới lên Drive (đè bản cũ). 2. *Thời gian chạy -> Thay đổi loại thời gian chạy -> T4 GPU*. 3. Chạy Cell 1 -> Cell 2 -> Cell 3.

**Cell 2 chạy lâu (~8 giờ tổng).** Mỗi lần chạy xong lưu ngay lên Drive. Colab ngắt hoặc bạn dừng lúc nào cũng được: mở lại -> chạy Cell 1 rồi Cell 2, phần đã xong tự bỏ qua. Cell 3 xem kết quả bất cứ lúc nào.

## Cell 1 — Chuẩn bị (code, checkpoint, CIFAR-100, Tiny-ImageNet). Lần đầu tải Tiny-ImageNet có thể mất lâu.

In [ ]:
import os, sys, glob, shutil, subprocess, torch
print('torch', torch.__version__, '| cuda ok:', torch.cuda.is_available())
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
RES = f'{DRIVE}/kan-vim-results/pilot'      # cùng thư mục với kết quả cũ -> cộng dồn
DDATA = f'{DRIVE}/kan-vim-data'
os.makedirs(RES, exist_ok=True); os.makedirs(DDATA, exist_ok=True); os.makedirs('/content/data', exist_ok=True)

# code
os.chdir('/content')
zips = sorted(glob.glob(f'{DRIVE}/**/kan-vim*.zip', recursive=True), key=os.path.getmtime)
assert zips, 'Không tìm thấy kan-vim.zip trên Drive'
print('zip:', zips[-1])
shutil.rmtree('/content/kan-vim', ignore_errors=True)
subprocess.run(['unzip', '-q', '-o', zips[-1], '-d', '/content'], check=True)
os.chdir('/content/kan-vim'); sys.path.insert(0, '/content/kan-vim')

# checkpoint
from huggingface_hub import hf_hub_download
shutil.copy(hf_hub_download('hustvl/Vim-tiny-midclstok', 'vim_t_midclstok_76p1acc.pth'), '/content/kan-vim/vim_t.pth')

# CIFAR-100 (từ Drive)
if os.path.isdir(f'{DDATA}/cifar-100-python'):
    shutil.copytree(f'{DDATA}/cifar-100-python', '/content/data/cifar-100-python', dirs_exist_ok=True)
else:
    from torchvision import datasets
    datasets.CIFAR100('/content/data', download=True)
    shutil.copytree('/content/data/cifar-100-python', f'{DDATA}/cifar-100-python', dirs_exist_ok=True)
print('CIFAR-100 ok')

# Tiny-ImageNet: lưu file zip trên Drive, giải nén vào máy Colab (nhanh hơn copy 120k file)
if not os.path.isdir('/content/data/tiny-imagenet-200'):
    if not os.path.exists(f'{DDATA}/tiny-imagenet-200.zip'):
        print('Tải Tiny-ImageNet (~240 MB, chỉ lần đầu)...')
        subprocess.run(['wget', '-q', '-O', '/content/tiny.zip', 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'], check=True)
        shutil.copy('/content/tiny.zip', f'{DDATA}/tiny-imagenet-200.zip')
    else:
        shutil.copy(f'{DDATA}/tiny-imagenet-200.zip', '/content/tiny.zip')
    subprocess.run(['unzip', '-q', '-o', '/content/tiny.zip', '-d', '/content/data'], check=True)
print('Tiny-ImageNet ok:', os.path.isdir('/content/data/tiny-imagenet-200'))
print('Kết quả đã có trên Drive:', len(glob.glob(f'{RES}/*.json')), 'lần chạy')

## Cell 2 — Chạy kế hoạch (tự bỏ qua phần đã xong)
| Đợt | Nội dung | Số lần | Thời gian ước tính |
|---|---|---|---|
| A | CIFAR-100, giữ 75% và 25%, seed 0 | 12 | ~2,5 giờ |
| B | CIFAR-100, giữ 50%, seed 1 và 2 (+ baseline) | 14 | ~3,5 giờ |
| C | Tiny-ImageNet, giữ 50%, seed 0 (+ baseline) | 7 | ~2 giờ |

In [ ]:
import os, glob, subprocess, time
os.chdir('/content/kan-vim')
M = ['kan_pool', 'mlp_pool', 'avg_pool', 'max_pool', 'kan_prune', 'mlp_prune']
PLAN = []
# Đợt A — CIFAR-100, mức 75% và 25%, seed 0                      (RQ1, RQ4)   ~2,5 giờ
PLAN += [('cifar100', r, q, 0) for q in (0.75, 0.25) for r in M]
# Đợt B — CIFAR-100, mức 50%, seed 1 và 2 (+ baseline)             (RQ2, RQ3: mean ± std)  ~3,5 giờ
PLAN += [('cifar100', 'identity', 1.0, s) for s in (1, 2)] + [('cifar100', r, 0.5, s) for s in (1, 2) for r in M]
# Đợt C — Tiny-ImageNet, mức 50%, seed 0 (+ baseline)              (RQ5)   ~2 giờ
PLAN += [('tiny-imagenet', 'identity', 1.0, 0)] + [('tiny-imagenet', r, 0.5, 0) for r in M]

COMMON = (f'--data_root /content/data --backbone tiny --ckpt vim_t.pth --epochs 1 --train_subset 5000 '
          f'--test_subset 2000 --batch 32 --grad_ckpt --no_bench --out {RES}')
todo = [p for p in PLAN if not os.path.exists(f'{RES}/{p[0]}_tiny_{p[1]}_r{p[2]}_s{p[3]}.json')]
print(f'Kế hoạch {len(PLAN)} lần chạy | đã xong {len(PLAN) - len(todo)} | còn {len(todo)}', flush=True)
for i, (ds, r, q, s) in enumerate(todo, 1):
    t = time.time()
    print(f'[{i}/{len(todo)}] {ds} | {r} | giữ {q:.0%} | seed {s}', flush=True)
    p = subprocess.run(f'python train.py --dataset {ds} --reducer {r} --ratio {q} --seed {s} {COMMON}',
                       shell=True, capture_output=True, text=True)
    lines = [l for l in (p.stdout + p.stderr).splitlines() if l.startswith(('epoch', 'saved', 'Traceback')) or 'Error' in l]
    for l in lines[-4:]:
        print('   ', l[:220])
    print(f'    {(time.time() - t) / 60:.1f} phút', flush=True)
print('XONG TOÀN BỘ KẾ HOẠCH')

## Cell 3 — Bảng và biểu đồ kết quả (chạy lúc nào cũng được)

In [ ]:
%cd /content/kan-vim
!python scripts/report.py {RES} > /dev/null
from IPython.display import Markdown, Image, display
import glob, os
txt = open(f'{RES}/report.md', encoding='utf-8').read()
display(Markdown('\n'.join(l for l in txt.splitlines() if not l.startswith('!['))))
for f in sorted(glob.glob(f'{RES}/fig_*.png')):
    print(os.path.basename(f)); display(Image(f))
print('Bảng tổng hợp:', f'{RES}/results_summary.csv')